# 01 — Uso de APIs LLM

**Módulo:** EAI_07 — IA Generativa  
**Submódulo:** 02_Modelos_PreTreinados

---

## O que você vai aprender

- Como funciona uma chamada de API para um LLM
- A estrutura de **mensagens** (system / user / assistant)
- Como usar o `llm_factory.py` com qualquer provider
- Parâmetros importantes: `temperature`, `max_tokens`
- Diferença entre resposta **completa** e **streaming**

---

## Pré-requisito

Antes de executar, confira se o `.env` na raiz do EAI_07 está configurado:  
```
LLM_PROVIDER=deepseek
LLM_MODEL=deepseek-chat
DEEPSEEK_API_KEY=sk-...
```

## 1. Setup — importando o factory compartilhado

In [1]:
import sys
import os

# Aponta para a raiz do EAI_07 (um nível acima de 02_Modelos_PreTreinados/)
sys.path.append(os.path.abspath('..'))

from shared.llm_factory import chat, chat_stream, get_provider_info

# Confirma qual provider está ativo
info = get_provider_info()
print(f"Provider : {info['provider']}")
print(f"Modelo   : {info['model']}")
print(f"Temp     : {info['temperature']}")
print(f"MaxTokens: {info['max_tokens']}")

Provider : deepseek
Modelo   : deepseek-chat
Temp     : 0.2
MaxTokens: 2048


## 2. Primeira chamada — o básico

Uma chamada de API tem sempre esta estrutura:

```
┌─────────────────────────────────────────┐
│  system   → quem o modelo deve ser      │
│  user     → o que você está pedindo     │
│  assistant→ resposta do modelo          │
└─────────────────────────────────────────┘
```

In [2]:
# Chamada mais simples possível — só o prompt
resposta = chat("O que é um Large Language Model? Responda em 3 linhas.")
print(resposta)

Um **Large Language Model (LLM)** é um modelo de inteligência artificial treinado com enormes volumes de texto para compreender e gerar linguagem humana. Ele funciona prevendo a próxima palavra mais provável em uma sequência, permitindo tarefas como tradução, redação e conversação. Exemplos notáveis incluem o GPT-4 da OpenAI e o Gemini do Google.


In [3]:
# Agora com system prompt — o model assume um papel específico
resposta = chat(
    prompt="O que é um Large Language Model?",
    system="Você é um professor de IA que explica conceitos complexos "
           "usando analogias do dia a dia. Seja didático e breve."
)
print(resposta)

Imagine que um **Large Language Model (LLM)** é como um **super chef de cozinha que nunca provou comida, mas leu todas as receitas, livros de culinária e críticas gastronômicas do mundo**.

- **Treinamento**: Ele "lê" trilhões de palavras (textos da internet, livros, artigos).  
- **Memória**: Não decora receitas, mas aprende **padrões**: "alho e cebola frequentemente aparecem juntos", "sobremesas precisam de açúcar", "críticas positivas usam palavras como 'delicioso'".  
- **Funcionamento**: Quando você pede "Escreva um poema sobre café", ele combina padrões de "poema" (ritmo, rimas) + "café" (aroma, energia, xícaras) para gerar algo novo.  
- **Limite**: Ele não entende o sabor do café, só sabe **descrevê-lo com base no que leu**. Pode inventar receitas absurdas se os padrões forem distorcidos.

**Resumindo**: É um sistema estatístico gigante que prevê a próxima palavra mais provável, baseado em padrões linguísticos aprendidos. Parece inteligente, mas é como um **papagaio ultra-infor

## 3. Parâmetro `temperature` — criatividade vs precisão

| Temperature | Comportamento | Uso ideal |
|---|---|---|
| `0.0` | Determinístico, sempre igual | Código, dados estruturados |
| `0.2` | Focado, pouca variação | Q&A técnico (nosso padrão) |
| `0.7` | Criativo, mais variado | Texto criativo, brainstorm |
| `1.0` | Muito criativo, imprevisível | Poesia, ficção |

In [4]:
prompt_teste = "Me dê uma palavra para descrever inteligência artificial."

print("=== Temperature 0.0 (determinístico) ===")
for i in range(3):
    r = chat(prompt_teste, temperature=0.0)
    print(f"  [{i+1}] {r.strip()}")

print()
print("=== Temperature 1.0 (criativo) ===")
for i in range(3):
    r = chat(prompt_teste, temperature=1.0)
    print(f"  [{i+1}] {r.strip()}")

=== Temperature 0.0 (determinístico) ===
  [1] **Alquimia** 

Porque a IA transforma dados brutos (o "chumbo" digital) em insights e ações valiosas (o "ouro" do conhecimento), através de processos complexos e muitas vezes misteriosos, assim como os antigos alquimistas buscavam.
  [2] **Alquimia** 

Porque a IA transforma dados brutos (o "chumbo" digital) em insights e ações valiosas (o "ouro" do conhecimento), através de processos complexos e muitas vezes misteriosos, assim como os antigos alquimistas buscavam.
  [3] **Alquimia** 

Porque a IA transforma dados brutos (o "chumbo" digital) em insights e ações valiosas (o "ouro" do conhecimento), através de processos complexos e muitas vezes misteriosos, assim como os antigos alquimistas buscavam.

=== Temperature 1.0 (criativo) ===
  [1] **Adaptativa** 

Essa palavra captura a essência da IA: a capacidade de aprender, ajustar-se e evoluir com base em dados e experiências, sem ser explicitamente reprogramada para cada nova situação.
  [2]

## 4. Parâmetro `max_tokens` — controlando o tamanho da resposta

**Token ≠ palavra**, mas como regra prática:
- 1 palavra em inglês ≈ 1 token  
- 1 palavra em português ≈ 1.3 tokens  
- 100 tokens ≈ 75 palavras

In [5]:
prompt_tokens = "Explique o conceito de atenção (attention) em redes neurais."

print("=== max_tokens=50 (resposta curta) ===")
r_curta = chat(prompt_tokens, max_tokens=50)
print(r_curta)
print(f"\n→ Caracteres: {len(r_curta)}")

print()
print("=== max_tokens=300 (resposta completa) ===")
r_longa = chat(prompt_tokens, max_tokens=300)
print(r_longa)
print(f"\n→ Caracteres: {len(r_longa)}")

=== max_tokens=50 (resposta curta) ===
## Conceito de Atenção (Attention) em Redes Neurais

### Ideia Central
A atenção é um mecanismo que permite que uma rede neural **focalize seletivamente** em partes específicas da entrada ao processar

→ Caracteres: 200

=== max_tokens=300 (resposta completa) ===
## Conceito de Atenção (Attention) em Redes Neurais

### Ideia Central
A atenção é um mecanismo que permite que uma rede neural **focalize seletivamente** em partes específicas da entrada ao processar informações, imitando a atenção cognitiva humana. Em vez de tratar todas as entradas com igual importância, o modelo aprende a **atribuir pesos diferentes** a diferentes elementos.

### Analogia Intuitiva
Imagine ler um texto longo para responder uma pergunta específica. Você não precisa processar cada palavra com a mesma intensidade - seu cérebro **foca nas partes relevantes** e ignora informações menos importantes. A atenção mecaniza esse processo para redes neurais.

### Problema que Reso

## 5. Streaming — tokens em tempo real

Em vez de esperar a resposta completa, o streaming entrega **token a token** — ideal para interfaces de chat onde o usuário vê o texto sendo gerado em tempo real.

In [6]:
print("Resposta com streaming:\n")

for chunk in chat_stream(
    prompt="Liste 5 aplicações práticas de LLMs no mundo real, uma por linha.",
    system="Seja conciso e direto."
):
    print(chunk, end="", flush=True)

print("\n\n[stream finalizado]")

Resposta com streaming:

1. **Assistentes virtuais** para atendimento ao cliente e suporte automatizado.
2. **Tradução automática** de textos em tempo real entre diversos idiomas.
3. **Geração de conteúdo** para marketing, como redação de e-mails e artigos.
4. **Resumo de documentos** longos, extraindo informações-chave de forma rápida.
5. **Tutoriais educacionais** personalizados e explicações adaptadas ao nível do aluno.

[stream finalizado]


## 6. Estrutura real da API — o que acontece por baixo

O `llm_factory.py` abstrai isso, mas é importante entender a estrutura real que vai para a API:

In [7]:
# Chamada direta ao SDK OpenAI (como o DeepSeek funciona por baixo)
# Isso é o que o llm_factory.py faz internamente

from openai import OpenAI
from dotenv import load_dotenv

load_dotenv('../.env')

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "Você é um assistente técnico."},
        {"role": "user",   "content": "O que é embeddings em NLP?"}
    ],
    temperature=0.2,
    max_tokens=200
)

# Estrutura completa da resposta
print("=== Resposta completa do objeto ===")
print(f"ID      : {response.id}")
print(f"Modelo  : {response.model}")
print(f"Tokens  : prompt={response.usage.prompt_tokens}, "
      f"resposta={response.usage.completion_tokens}, "
      f"total={response.usage.total_tokens}")
print()
print("=== Texto da resposta ===")
print(response.choices[0].message.content)

=== Resposta completa do objeto ===
ID      : da8e4d50-ae65-4871-bca2-ac3f494b6caf
Modelo  : deepseek-chat
Tokens  : prompt=20, resposta=200, total=220

=== Texto da resposta ===
**Embeddings em NLP** (Processamento de Linguagem Natural) são representações vetoriais (numéricas) de palavras, frases ou documentos que capturam seu significado semântico em um espaço multidimensional. Eles são fundamentais para que modelos de machine learning "entendam" e processem linguagem humana.

---

### **Principais Características:**

1. **Representação Densa**  
   - Convertem palavras em vetores de números reais (ex: [0.2, -0.5, 0.8, ...]), ao contrário de representações esparsas como *one-hot encoding*.
   - Dimensões típicas: 50 a 1000 (ex: Word2Vec usa 300 dimensões).

2. **Capturam Relações Semânticas**  
   - Palavras com significados similares têm vetores próximos no espaço (ex: "rei


## 7. Estimativa de custo

Uma das vantagens do DeepSeek é o custo muito baixo. Vamos calcular:

In [8]:
# Preços DeepSeek (deepseek-chat) em março/2025
PRECO_INPUT_POR_MILHAO  = 0.27   # USD por 1M tokens de entrada
PRECO_OUTPUT_POR_MILHAO = 1.10   # USD por 1M tokens de saída

def estimar_custo(tokens_input: int, tokens_output: int) -> dict:
    custo_input  = (tokens_input  / 1_000_000) * PRECO_INPUT_POR_MILHAO
    custo_output = (tokens_output / 1_000_000) * PRECO_OUTPUT_POR_MILHAO
    total = custo_input + custo_output
    return {
        "custo_input_usd" : round(custo_input,  6),
        "custo_output_usd": round(custo_output, 6),
        "total_usd"       : round(total,        6),
        "total_brl"       : round(total * 5.0,  5),  # câmbio aproximado
    }

# Exemplos práticos
cenarios = [
    ("1 pergunta simples",        200,    150),
    ("1 pergunta com contexto",  1000,    500),
    ("100 perguntas/dia",       20000,  15000),
    ("Indexar todo o projeto",  50000,   1000),
]

print(f"{'Cenário':<30} {'Input':>8} {'Output':>8} {'Total USD':>12} {'Total BRL':>10}")
print("-" * 75)
for nome, inp, out in cenarios:
    c = estimar_custo(inp, out)
    print(f"{nome:<30} {inp:>8,} {out:>8,} "
          f"$ {c['total_usd']:>9.6f}  R$ {c['total_brl']:>7.5f}")

Cenário                           Input   Output    Total USD  Total BRL
---------------------------------------------------------------------------
1 pergunta simples                  200      150 $  0.000219  R$ 0.00110
1 pergunta com contexto           1,000      500 $  0.000820  R$ 0.00410
100 perguntas/dia                20,000   15,000 $  0.021900  R$ 0.10950
Indexar todo o projeto           50,000    1,000 $  0.014600  R$ 0.07300


## 8. Conversa multi-turno — mantendo contexto

LLMs são **stateless** — cada chamada é independente. Para manter contexto, enviamos o histórico completo a cada chamada.

In [9]:
def conversar(historico: list, nova_mensagem: str, system: str = None) -> tuple[str, list]:
    """Adiciona nova mensagem ao histórico e retorna resposta + histórico atualizado."""
    from openai import OpenAI

    client = OpenAI(
        api_key=os.getenv("DEEPSEEK_API_KEY"),
        base_url="https://api.deepseek.com"
    )

    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.extend(historico)
    messages.append({"role": "user", "content": nova_mensagem})

    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=messages,
        temperature=0.2,
        max_tokens=300
    )

    resposta = response.choices[0].message.content

    # Atualiza histórico
    historico_novo = historico + [
        {"role": "user",      "content": nova_mensagem},
        {"role": "assistant", "content": resposta}
    ]
    return resposta, historico_novo


# Simulando uma conversa sobre o projeto
system = "Você é um assistente técnico especializado em Machine Learning e IA."
historico = []

perguntas = [
    "O que é RAG?",
    "Como isso se diferencia de fine-tuning?",
    "Qual dos dois é mais barato de implementar?"
]

for pergunta in perguntas:
    print(f"👤 {pergunta}")
    resposta, historico = conversar(historico, pergunta, system)
    print(f"🤖 {resposta}")
    print(f"   [histórico: {len(historico)} mensagens]\n")

👤 O que é RAG?
🤖 **RAG (Retrieval-Augmented Generation)** é uma arquitetura avançada de IA que combina **recuperação de informações** com **geração de texto**, criando sistemas mais precisos, atualizáveis e com menor tendência a "alucinações" (respostas inventadas).

---

### **Como funciona (de forma simplificada):**
1. **Recuperação (Retrieval):**  
   Quando recebe uma pergunta, o sistema busca em uma base de dados externa (documentos, artigos, PDFs, etc.) os trechos mais relevantes para o contexto.

2. **Aumentação (Augmentation):**  
   Esses trechos recuperados são inseridos como contexto adicional no **prompt** enviado ao modelo de linguagem (LLM).

3. **Geração (Generation):**  
   O LLM gera uma resposta **baseada no contexto fornecido**, em vez de confiar apenas em seu conhecimento interno (treinamento original).

---

### **Por que é importante?**
- **Reduz alucinações:** O modelo "ancora" suas respostas em fontes externas verificáveis.
- **Atualização fácil:** Para adiciona

---

## Resumo

| Conceito | O que aprendemos |
|---|---|
| **Mensagens** | Estrutura `system` / `user` / `assistant` |
| **`temperature`** | Controla criatividade (0 = determinístico, 1 = criativo) |
| **`max_tokens`** | Limita o tamanho da resposta |
| **Streaming** | Entrega tokens em tempo real com `chat_stream()` |
| **Multi-turno** | Contexto mantido enviando histórico completo |
| **Custo** | DeepSeek é muito barato — centavos por centenas de chamadas |

---